In [1]:
import sys
import torch
import os

print("--- 系统与环境检查结果 ---")

# 1. 检查 Python 版本
print(f"Python 版本: {sys.version.split()[0]}")

# 2. 检查 PyTorch 是否安装及版本
try:
    print(f"PyTorch 版本: {torch.__version__}")
    # 检查是否有可用的 GPU (CUDA)
    if torch.cuda.is_available():
        print(f"GPU 可用: {torch.cuda.get_device_name(0)}")
        print(f"CUDA 版本: {torch.version.cuda}")
    else:
        print("提示: 未检测到可用的 GPU，将使用 CPU 运行（速度会较慢）。")
except ImportError:
    print("错误: 未检测到 PyTorch。请先在终端中运行 'pip install torch torchvision' 安装。")

# 3. 检查当前工作目录是否正确
current_path = os.getcwd()
if 'LAMSC-master' in current_path:
    print(f"当前路径正确: {current_path}")
else:
    print(f"注意: 当前路径为 {current_path}，请确保 Notebook 创建在 LAMSC-master 文件夹内。")

print("-----------------------------")

--- 系统与环境检查结果 ---
Python 版本: 3.9.25
PyTorch 版本: 2.5.1
提示: 未检测到可用的 GPU，将使用 CPU 运行（速度会较慢）。
当前路径正确: C:\Users\Vickie-77\Desktop\LAMSC-master\LAMSC-master
-----------------------------


In [2]:
import os
import requests
from pathlib import Path

print("--- 开始下载预训练权重 ---")

# 1. 在当前目录下创建 checkpoints 文件夹
checkpoint_dir = Path("checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)

# 2. 定义权重文件保存路径
# LAMSC 论文中使用的 SAM 模型通常是最大版本的 ViT-H
save_path = checkpoint_dir / "sam_vit_h_4b8939.pth"

# 如果文件不存在，则开始下载
if not save_path.exists():
    print("正在从官方服务器下载 SAM 权重文件 (约 2.4GB)，请耐心等待...")
    
    # Meta 官方提供的直接下载链接
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # 确保下载成功
        
        total_size = int(response.headers.get('content-length', 0))
        downloaded_size = 0
        
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded_size += len(chunk)
                if total_size > 0:
                    progress = (downloaded_size / total_size) * 100
                    print(f"下载进度: {progress:.1f}%", end="\r")
        
        print(f"\n权重文件下载成功！已保存至: {save_path.absolute()}")
    except requests.exceptions.RequestException as e:
        print(f"\n下载失败: {e}")
        print("请检查网络连接，或者稍后手动下载后放入 checkpoints 文件夹。")
else:
    print(f"权重文件已存在，无需重复下载: {save_path.absolute()}")

print("-----------------------------")

--- 开始下载预训练权重 ---
正在从官方服务器下载 SAM 权重文件 (约 2.4GB)，请耐心等待...
下载进度: 100.0%
权重文件下载成功！已保存至: C:\Users\Vickie-77\Desktop\LAMSC-master\LAMSC-master\checkpoints\sam_vit_h_4b8939.pth
-----------------------------


In [4]:
# 安装 LAMSC 项目所需的所有依赖包
# 注意：torch 如果已安装会跳过，没安装会下载（较大，请耐心等待）

print("=== 开始安装依赖包 ===")

# 1. 安装基础依赖（numpy, opencv, matplotlib, Pillow, torchsummary）
%pip install numpy opencv-python matplotlib Pillow torchsummary -q

# 3. 安装 SAM（Segment Anything Model）库
# 这个库包含 SAM 模型的核心代码
%pip install segment-anything -q

print("=== 基础依赖安装完成 ===")
print("提示：如果上面有任何报错，请截图发给我。")

=== 开始安装依赖包 ===
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
=== 基础依赖安装完成 ===
提示：如果上面有任何报错，请截图发给我。


In [6]:
# 验证所有依赖是否安装成功
print("=== 依赖安装验证 ===\n")

dependencies = [
    ("numpy", "numpy"),
    ("opencv-python", "cv2"),
    ("matplotlib", "matplotlib"),
    ("Pillow", "PIL"),
    ("torchsummary", "torchsummary"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("segment-anything", "segment_anything"),
]

all_ok = True
for name, import_name in dependencies:
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "未知版本")
        print(f"  OK  {name} ({import_name}) - 版本: {version}")
    except ImportError:
        print(f"  FAIL {name} ({import_name}) - 未安装！")
        all_ok = False

# 修复：补充完整的 for 循环内容
sub_deps = [
    ("pycocotools", "pycocotools"),
    ("onnxruntime", "onnxruntime")
]

for name, import_name in sub_deps:
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "未知版本")
        print(f"  OK  {name} ({import_name}) - 版本: {version}")
    except ImportError:
        print(f"  WARN {name} ({import_name}) - 未安装（SAM 可能需要）")

# 检查 GPU
try:
    import torch
    if torch.cuda.is_available():
        print(f"\nGPU 可用: {torch.cuda.get_device_name(0)}")
    else:
        print("\n提示: 未检测到 GPU，将使用 CPU 运行（速度较慢）")
except:
    pass

print("\n===================")
if all_ok:
    print("所有核心依赖已安装成功！可以继续运行 SKB.py")
else:
    print("有部分依赖未安装成功，请检查上面的 FAIL 项并重新安装")

=== 依赖安装验证 ===

  OK  numpy (numpy) - 版本: 2.0.1
  OK  opencv-python (cv2) - 版本: 5.0.0
  OK  matplotlib (matplotlib) - 版本: 3.9.4
  OK  Pillow (PIL) - 版本: 11.1.0
  OK  torchsummary (torchsummary) - 版本: 未知版本
  OK  torch (torch) - 版本: 2.5.1
  OK  torchvision (torchvision) - 版本: 0.20.1
  OK  segment-anything (segment_anything) - 版本: 未知版本
  WARN pycocotools (pycocotools) - 未安装（SAM 可能需要）
  WARN onnxruntime (onnxruntime) - 未安装（SAM 可能需要）

提示: 未检测到 GPU，将使用 CPU 运行（速度较慢）

所有核心依赖已安装成功！可以继续运行 SKB.py


In [9]:
%run SKB.py

D:\Anaconda\envs\ai_course\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


data/segments/example\2008_000016\0000.jpg
data/segments/example\2008_000016\0001.jpg
data/segments/example\2008_000016\0002.jpg
data/segments/example\2008_000016\0003.jpg
data/segments/example\2008_000016\0004.jpg


D:\Anaconda\envs\ai_course\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


data/segments/example\2008_000026\0000.jpg
data/segments/example\2008_000026\0001.jpg
data/segments/example\2008_000026\0002.jpg
data/segments/example\2008_000026\0003.jpg
data/segments/example\2008_000026\0004.jpg


D:\Anaconda\envs\ai_course\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


data/segments/example\2008_000033\0000.jpg
data/segments/example\2008_000033\0001.jpg
data/segments/example\2008_000033\0002.jpg
data/segments/example\2008_000033\0003.jpg
data/segments/example\2008_000033\0004.jpg


... (36 outputs left out)

D:\Anaconda\envs\ai_course\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


data/segments/example\2008_001691\0000.jpg
data/segments/example\2008_001691\0001.jpg
data/segments/example\2008_001691\0002.jpg
data/segments/example\2008_001691\0003.jpg
data/segments/example\2008_001691\0004.jpg


D:\Anaconda\envs\ai_course\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


data/segments/example\2008_001719\0000.jpg
data/segments/example\2008_001719\0001.jpg
data/segments/example\2008_001719\0002.jpg
data/segments/example\2008_001719\0003.jpg
data/segments/example\2008_001719\0004.jpg


In [10]:
import os
from pathlib import Path

root = Path("data/segments")
if not root.exists():
    print("未找到 data/segments，请确认 SKB.py 的输出路径")
else:
    total = 0
    for sub in sorted(root.iterdir()):
        if sub.is_dir():
            files = list(sub.glob("*"))
            total += len(files)
            print(f"{sub.name}: {len(files)} 个文件")
    print(f"\n总计: {total} 个文件")

example: 23 个文件

总计: 23 个文件


In [11]:
%run ASI.py

AttributeError: module 'torch._C' has no attribute '_cuda_setDevice'